In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [5]:
df.head()   # 34 features, 1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [6]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [7]:
df.shape

(898, 35)

In [8]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [9]:
df["Class"].unique()  # how many unique class we have

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [13]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [20]:
## create tensors = first stape

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [21]:
# Create datasets = second step

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [22]:
# create dataloader = 3rd stape

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [30]:
# start to build our model

class ANN(nn.Module):
    def __init__(self):     # define construct unit
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),  # first hidden layer
            nn.ReLU(),
            nn.Linear(64, 64),  # second hidden layer
            nn.ReLU(),
            nn.Linear(64, 7)    # final output layer
            
        )

    def forward(self, x):
        return self.model(x)

In [31]:
# ready the model, criteria, optimizer
model = ANN()

# loss & optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [32]:
# Traning the NN

epochs = 100
for epoch in range(epochs):
    model.train()      # add the model on training mode

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)      # it is a Forward propagation
        loss = criteria(outputs, yb)    # criteria is basicaly loss fnx (loss are compute in this state + compute gtadient)
        loss.backward()       # it is a backward propagation
        optimizer.step()    # param update stape

        running_loss += loss.item()    # add loss in rinning loss

    train_loss = running_loss / len(train_loader)   # train_loss is the avg loss per batch in one epoch

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.694174885749817
epoch = 2/100, loss = 1.0882439898407978
epoch = 3/100, loss = 0.6917791392492212
epoch = 4/100, loss = 0.545721006134282
epoch = 5/100, loss = 0.44306523061316944
epoch = 6/100, loss = 0.3831003256466078
epoch = 7/100, loss = 0.3358103432085203
epoch = 8/100, loss = 0.3012211983618529
epoch = 9/100, loss = 0.28171804482522217
epoch = 10/100, loss = 0.26276530973289325
epoch = 11/100, loss = 0.2518291227195574
epoch = 12/100, loss = 0.23165222160194232
epoch = 13/100, loss = 0.21658065170049667
epoch = 14/100, loss = 0.2068627988514693
epoch = 15/100, loss = 0.20795618613129077
epoch = 16/100, loss = 0.194042534607908
epoch = 17/100, loss = 0.1872910079748734
epoch = 18/100, loss = 0.1766521464223447
epoch = 19/100, loss = 0.1701634416113729
epoch = 20/100, loss = 0.16212920034709183
epoch = 21/100, loss = 0.1579305442131084
epoch = 22/100, loss = 0.15288275415482727
epoch = 23/100, loss = 0.14617126354056856
epoch = 24/100, loss = 0.145467702137

In [35]:
# Evaluate
model.eval()

total = 0   # total val = witch amount of total value are their in testing data
correct = 0   # which amount he predict currect

# run for all the batchs
with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)  # output values type [0.2, 0.5, 1.3, -0.5, ..] - 7 vals
        _, predicted = torch.max(outputs, 1)

        correct = (predicted == yb).sum().item()  # calculate sum of yb equal all predict values(this is a tensor) and convert this in a python.
        total += yb.size(0)    # actual samples in each batch

print("total vals: ", total)
print("correct vals: ", correct)
print("accuracy: ", correct/total * 100)

total vals:  180
correct vals:  19
accuracy:  10.555555555555555
